# EDA: UCI Gas Sensor Array Drift Dataset
Run `pip install -r ../requirements.txt` first, and place `batch*.dat` in `../data/raw/` if you have the real dataset (otherwise this uses synthetic demo data).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from data_loader import load_all_batches, make_synthetic_demo_dataset, FEATURE_NAMES
from preprocessing import clean

RAW_DIR = '../data/raw'
if os.path.isdir(RAW_DIR) and any(f.startswith('batch') for f in os.listdir(RAW_DIR)):
    df = load_all_batches(RAW_DIR)
else:
    print('No real data found -- using synthetic demo dataset')
    df = make_synthetic_demo_dataset(6000)
df = clean(df, FEATURE_NAMES)
df.shape

In [ ]:
# Gas class distribution per batch
pd.crosstab(df['batch'], df['gas_class'])

In [ ]:
# Visualize drift: mean of sensor 1's descriptors, per batch
sensor1_cols = FEATURE_NAMES[0:8]
batch_means = df.groupby('batch')[sensor1_cols].mean().mean(axis=1)
plt.figure(figsize=(8,4))
plt.plot(batch_means.index, batch_means.values, marker='o')
plt.xlabel('Batch (time)')
plt.ylabel('Sensor 1 mean response')
plt.title('Drift across batches -- sensor 1')
plt.show()

In [ ]:
# PCA projection colored by batch, to visually see the drift manifold
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

X = StandardScaler().fit_transform(df[FEATURE_NAMES])
pcs = PCA(n_components=2).fit_transform(X)
plt.figure(figsize=(6,5))
sc = plt.scatter(pcs[:,0], pcs[:,1], c=df['batch'], cmap='viridis', s=8, alpha=0.6)
plt.colorbar(sc, label='batch')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.title('PCA of sensor features, colored by batch (drift visualization)')
plt.show()

## Next: run the full pipeline
See `../src/main.py` for preprocessing, fault injection, feature engineering, model training and evaluation.